# 22 - Prepare Replay Dataset

## Purpose

Prepare a dedicated replay dataset for Kafka streaming.

This notebook does **not** generate timestamps.

The replay producer is responsible for assigning realistic event timestamps during streaming.

### Input

data/processed/creditcard_feature_engineered.csv

### Output

data/replay/creditcard_replay.csv

### Operations

- Validate dataset
- Shuffle transactions
- Generate Replay ID
- Generate Replay Batch
- Export replay dataset

# ============================================
# Section 1 - Imports
# ============================================

In [ ]:


import os
import logging

from datetime import datetime, timedelta

import numpy as np
import pandas as pd

print("Libraries imported successfully.")

# ============================================
# Section 2 - Configuration
# ============================================

In [ ]:
INPUT_DATASET = "../data/processed/creditcard_feature_engineered.csv"

OUTPUT_DIRECTORY = "../data/replay"

OUTPUT_DATASET = os.path.join(
    OUTPUT_DIRECTORY,
    "creditcard_replay.csv"
)

LOG_DIRECTORY = "../logs"

LOG_FILE = os.path.join(
    LOG_DIRECTORY,
    "prepare_replay_dataset.log"
)

RANDOM_STATE = 42

ROWS_PER_BATCH = 1000

os.makedirs(OUTPUT_DIRECTORY, exist_ok=True)
os.makedirs(LOG_DIRECTORY, exist_ok=True)

print("Configuration loaded.")

# ============================================
# Section 3 - Logging
# ============================================

In [ ]:
logging.basicConfig(

    filename=LOG_FILE,

    level=logging.INFO,

    format="%(asctime)s | %(levelname)s | %(message)s",

    filemode="a"

)

logging.info("="*80)
logging.info("Replay Dataset Preparation Started")
logging.info("="*80)

print("Logging initialized.")

# ============================================
# Section 4 - Verify Input Dataset
# ============================================

In [ ]:
if not os.path.exists(INPUT_DATASET):

    raise FileNotFoundError(
        f"Dataset not found:\n{INPUT_DATASET}"
    )

print("Dataset Found.")

logging.info("Dataset verified.")

# ============================================
# Section 5 - Load Dataset
# ============================================

In [ ]:

df = pd.read_csv(INPUT_DATASET)

print(f"Rows    : {len(df)}")
print(f"Columns : {len(df.columns)}")

logging.info(f"Rows Loaded : {len(df)}")
logging.info(f"Columns : {len(df.columns)}")

# ============================================
# Section 6 - Validate Dataset
# ============================================

In [ ]:
print("="*60)
print("Dataset Validation")
print("="*60)

print()

print("Missing Values")

print(df.isnull().sum().sum())

print()

print("Duplicate Rows")

print(df.duplicated().sum())

logging.info("Dataset validation completed.")

# ============================================
# Section 7 - Shuffle Dataset
# ============================================

In [ ]:


replay_df = (

    df.sample(

        frac=1,

        random_state=RANDOM_STATE

    )

    .reset_index(drop=True)

)

logging.info("Dataset shuffled.")

# ============================================
# Section 8 - Replay Metadata
# ============================================

In [ ]:


replay_df.insert(

    0,

    "Replay_ID",

    np.arange(1, len(replay_df)+1)

)

replay_df.insert(

    1,

    "Replay_Batch",

    ((replay_df["Replay_ID"]-1)//ROWS_PER_BATCH)+1

)

logging.info("Replay metadata created.")

# ============================================
# Section 9 - Save Replay Dataset
# ============================================

In [ ]:


replay_df.to_csv(

    OUTPUT_DATASET,

    index=False

)

logging.info("Replay dataset exported.")

# ============================================
# Section 10 - Validate Output
# ============================================

In [ ]:


saved_df = pd.read_csv(OUTPUT_DATASET)

assert len(saved_df) == len(replay_df)

assert "Replay_ID" in saved_df.columns

assert "Replay_Batch" in saved_df.columns

print()

print("Replay Dataset Validation Passed")

print()

print(saved_df.head())

logging.info("Replay dataset validated.")

# ============================================
# Section 11 - Summary
# ============================================

In [ ]:


print("="*70)

print("Replay Dataset Created Successfully")

print("="*70)

print(f"Rows                : {len(replay_df)}")

print(f"Columns             : {len(replay_df.columns)}")

print(f"Replay Batches      : {replay_df['Replay_Batch'].nunique()}")

print(f"Output File         : {OUTPUT_DATASET}")

print()

print("Timestamp Generation : Producer")

print("Streaming           : Kafka Producer")

print("="*70)

logging.info("Replay dataset preparation completed.")
logging.info("="*80)